# Tarea 1: Analizador de Sentimiento con TF-IDF + Regresión Logística

**Curso:** Procesamiento de Lenguaje Natural — UCB  
**Entrega:** Semana 4  
**Valor:** 10% de la nota final  

---

## Objetivo

Construir un clasificador de sentimiento binario (positivo / negativo) para reseñas en español utilizando:

- **Preprocesamiento de texto** (tokenización, lematización, eliminación de stopwords)
- **Representación vectorial TF-IDF**
- **Regresión Logística** como modelo de clasificación

## Dataset

Utilizaremos el dataset [`sepidmnorozy/Spanish_sentiment`](https://huggingface.co/datasets/sepidmnorozy/Spanish_sentiment) de Hugging Face, que contiene reseñas en español con etiquetas binarias de sentimiento.

- `label = 1` → Sentimiento positivo
- `label = 0` → Sentimiento negativo

## Rúbrica de Evaluación

| Sección | Puntos | Descripción |
|---------|:------:|-------------|
| **Parte 1:** Carga y EDA | 15 | Carga correcta del dataset, análisis exploratorio completo |
| **Parte 2:** Preprocesamiento | 20 | Pipeline de limpieza de texto bien implementado |
| **Parte 3:** TF-IDF | 15 | Vectorización correcta con parámetros justificados |
| **Parte 4:** Modelo | 20 | Entrenamiento de Regresión Logística con manejo de desbalance |
| **Parte 5:** Evaluación | 20 | Métricas completas, matriz de confusión, análisis de errores |
| **Parte 6:** Reflexión | 10 | Respuestas reflexivas y bien fundamentadas |
| **TOTAL** | **100** | |

---

## Configuración Inicial

Ejecuta esta celda para instalar las dependencias necesarias.

In [ ]:
# Instalación de dependencias (ejecutar solo una vez)
# !uv pip install datasets scikit-learn nltk spacy matplotlib seaborn pandas tqdm
# !python -m spacy download es_core_news_sm

In [ ]:
# Imports necesarios
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# NLP
import re
import spacy
import nltk
from nltk.corpus import stopwords

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Dataset
from datasets import load_dataset

# Configuración
nltk.download('stopwords', quiet=True)
nlp = spacy.load('es_core_news_sm')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Todo listo!')

---

## Parte 1: Carga de Datos y Análisis Exploratorio (15 pts)

### 1.1 Carga del dataset (5 pts)

Carga el dataset `sepidmnorozy/Spanish_sentiment` desde Hugging Face y conviértelo a un DataFrame de Pandas con las columnas `text` y `label`.

**Requisitos:**
- Cargar los splits `train`, `validation` y `test`
- Convertir cada split a un DataFrame
- Mostrar las dimensiones de cada split y las primeras 5 filas del split de entrenamiento

In [ ]:
# TODO: Carga el dataset desde Hugging Face
# Hint: usa load_dataset('sepidmnorozy/Spanish_sentiment')

ds = # Tu código aquí

# TODO: Convierte cada split a un DataFrame de Pandas
df_train = # Tu código aquí
df_val = # Tu código aquí
df_test = # Tu código aquí

# TODO: Muestra las dimensiones de cada split
print(f"Train: {df_train.shape}")
print(f"Validation: {df_val.shape}")
print(f"Test: {df_test.shape}")

# Muestra las primeras filas
df_train.head()

### 1.2 Análisis Exploratorio (10 pts)

Realiza un análisis exploratorio básico que incluya:

1. **Distribución de clases:** Gráfico de barras mostrando la cantidad de ejemplos positivos vs negativos en el set de entrenamiento
2. **Distribución de longitud de textos:** Histograma de la cantidad de palabras por reseña
3. **Estadísticas descriptivas:** Longitud promedio, mediana, mínima y máxima de las reseñas por clase

**Pregunta:** ¿El dataset está balanceado? ¿Qué implicaciones tiene esto para el entrenamiento del modelo?

In [ ]:
# TODO 1.2.1: Gráfico de distribución de clases
# Hint: usa value_counts() y plot con kind='bar'


In [ ]:
# TODO 1.2.2: Agrega columna con largo de texto (en palabras) y haz un histograma
# Hint: df_train['text'].str.split().str.len()


In [ ]:
# TODO 1.2.3: Estadísticas descriptivas por clase
# Hint: groupby('label') y describe()


**Tu respuesta sobre el desbalance:**

*[Escribe tu respuesta aquí]*

---

## Parte 2: Preprocesamiento de Texto (20 pts)

### 2.1 Función de preprocesamiento (15 pts)

Implementa una función `preprocesar_texto(texto)` que realice los siguientes pasos:

1. Convertir a minúsculas
2. Eliminar URLs, menciones (@usuario) y hashtags
3. Eliminar signos de puntuación y números
4. Tokenizar y lematizar usando spaCy (`es_core_news_sm`)
5. Eliminar stopwords en español
6. Eliminar tokens con menos de 2 caracteres

La función debe retornar un string con los tokens procesados separados por espacios.

In [ ]:
# Stopwords en español
STOPWORDS_ES = set(stopwords.words('spanish'))

def preprocesar_texto(texto: str) -> str:
    """
    Preprocesa un texto en español para análisis de sentimiento.
    
    Args:
        texto: El texto crudo de la reseña
        
    Returns:
        String con tokens lematizados, sin stopwords, en minúsculas
    """
    # TODO: Implementa los 6 pasos de preprocesamiento
    # Paso 1: Minúsculas
    
    # Paso 2: Eliminar URLs, menciones, hashtags
    
    # Paso 3: Eliminar puntuación y números
    
    # Paso 4: Tokenizar y lematizar con spaCy
    
    # Paso 5: Eliminar stopwords
    
    # Paso 6: Eliminar tokens cortos (< 2 caracteres)
    
    pass  # Reemplaza con tu implementación


# Prueba tu función con estos ejemplos:
ejemplos = [
    "¡Este hotel es INCREÍBLE! Lo recomiendo 100%.",
    "Pésimo servicio, no vuelvo nunca más... 😡",
    "El restaurante está bien, nada especial. Precios normales."
]

for ej in ejemplos:
    print(f"Original: {ej}")
    print(f"Procesado: {preprocesar_texto(ej)}")
    print()

### 2.2 Aplicar preprocesamiento al dataset (5 pts)

Aplica tu función de preprocesamiento a los tres splits del dataset. Crea una nueva columna llamada `text_clean`.

**Nota:** Este paso puede tomar algunos minutos. Muestra una barra de progreso o imprime cada 100 textos procesados.

In [ ]:
# TODO: Aplica preprocesar_texto a los tres DataFrames
# Crea la columna 'text_clean' en cada uno
# Hint: puedes usar .apply() o un loop con tqdm

from tqdm import tqdm
tqdm.pandas()

# df_train['text_clean'] = ...
# df_val['text_clean'] = ...
# df_test['text_clean'] = ...

# Muestra ejemplos del resultado
df_train[['text', 'text_clean', 'label']].head(10)

---

## Parte 3: Vectorización TF-IDF (15 pts)

### 3.1 Crear la representación TF-IDF (10 pts)

Utiliza `TfidfVectorizer` de scikit-learn para convertir los textos preprocesados en vectores TF-IDF.

**Requisitos:**
- Ajusta (`fit`) el vectorizador **solo** con los datos de entrenamiento
- Transforma los tres splits con el mismo vectorizador
- Experimenta con los parámetros: `max_features`, `min_df`, `max_df`, `ngram_range`
- Justifica brevemente los parámetros elegidos

In [ ]:
# TODO: Crea y ajusta el vectorizador TF-IDF
# Importante: fit SOLO con train, transform en los tres splits

tfidf = TfidfVectorizer(
    # TODO: Define tus parámetros aquí
    # max_features=...,
    # min_df=...,
    # max_df=...,
    # ngram_range=...,
)

# TODO: Ajustar con train y transformar los tres splits
# X_train = ...
# X_val = ...
# X_test = ...

# Labels
# y_train = ...
# y_val = ...
# y_test = ...

print(f"Dimensiones X_train: {X_train.shape}")
print(f"Dimensiones X_val: {X_val.shape}")
print(f"Dimensiones X_test: {X_test.shape}")
print(f"Vocabulario: {len(tfidf.vocabulary_)} términos")

**Justificación de parámetros:**

*[Explica por qué elegiste esos valores para max_features, min_df, max_df y ngram_range]*

### 3.2 Inspección del vocabulario TF-IDF (5 pts)

Analiza el vocabulario generado:
1. Muestra los 20 términos con mayor valor IDF (más raros)
2. Muestra los 20 términos con menor valor IDF (más comunes)
3. ¿Hay algo que te sorprenda? ¿Algún término que no debería estar?

In [ ]:
# TODO: Inspecciona los valores IDF del vocabulario
# Hint: tfidf.idf_ contiene los valores IDF, tfidf.get_feature_names_out() los términos


**Tu análisis del vocabulario:**

*[Escribe tu respuesta aquí]*

---

## Parte 4: Entrenamiento del Modelo (20 pts)

### 4.1 Regresión Logística base (10 pts)

Entrena un modelo de Regresión Logística como baseline.

**Requisitos:**
- Usa `class_weight='balanced'` para manejar el desbalance de clases
- Entrena con `X_train` / `y_train`
- Evalúa en `X_val` / `y_val` para selección de hiperparámetros

In [ ]:
# TODO: Entrena el modelo de Regresión Logística
# Hint: LogisticRegression(class_weight='balanced', max_iter=1000)

modelo = # Tu código aquí

# TODO: Entrena el modelo

# TODO: Predice en el set de validación
y_val_pred = # Tu código aquí

# Muestra el reporte de clasificación en validación
print("=== Resultados en Validación ===")
print(classification_report(y_val, y_val_pred, target_names=['Negativo', 'Positivo']))

### 4.2 Experimentación con hiperparámetros (10 pts)

Experimenta con al menos **2** configuraciones diferentes y reporta los resultados.

**Ideas para experimentar:**
- Valor de regularización `C` (ej: 0.01, 0.1, 1, 10)
- Con y sin `class_weight='balanced'`
- Diferentes parámetros de TF-IDF (unigrams vs bigrams, más/menos features)
- Cambiar el `solver` (liblinear, lbfgs, saga)

Reporta los resultados en una tabla comparativa usando el set de **validación**.

In [ ]:
# TODO: Experimenta con al menos 2 configuraciones diferentes
# Crea una tabla comparativa de resultados

# Ejemplo de estructura:
resultados = []

configs = [
    # TODO: Define tus configuraciones
    # {'nombre': 'Config 1', 'C': ..., 'class_weight': ..., ...},
    # {'nombre': 'Config 2', 'C': ..., 'class_weight': ..., ...},
]

for config in configs:
    # TODO: Entrena y evalúa cada configuración
    pass

# TODO: Muestra tabla comparativa
# Hint: pd.DataFrame(resultados)

**Análisis de resultados:**

*[¿Cuál configuración dio mejores resultados? ¿Por qué crees que fue así?]*

---

## Parte 5: Evaluación Final en Test (20 pts)

### 5.1 Métricas finales (10 pts)

Usa tu **mejor modelo** (según validación) para predecir en el set de **test**.

Reporta:
- Accuracy
- Precision, Recall y F1 por clase
- F1 Macro (promedio de ambas clases)
- Matriz de confusión visualizada

In [ ]:
# TODO: Evalúa tu mejor modelo en el set de TEST

# Predicciones
y_test_pred = # Tu código aquí

# Reporte de clasificación
print("=== Resultados Finales en Test ===")
print(classification_report(y_test, y_test_pred, target_names=['Negativo', 'Positivo']))

# TODO: Calcula y muestra F1 Macro
f1_macro = # Tu código aquí
print(f"\nF1 Macro: {f1_macro:.4f}")

In [ ]:
# TODO: Visualiza la matriz de confusión
# Hint: ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, ...)


### 5.2 Análisis de errores (10 pts)

Examina los errores del modelo:

1. Muestra **5 falsos positivos** (predicho positivo, real negativo) con el texto original
2. Muestra **5 falsos negativos** (predicho negativo, real positivo) con el texto original
3. Muestra los **10 términos más importantes** para cada clase (los coeficientes más altos y más bajos del modelo)
4. Analiza: ¿Por qué crees que el modelo se equivocó en esos casos?

In [ ]:
# TODO 5.2.1: Muestra falsos positivos y falsos negativos
# Hint: compara y_test con y_test_pred y busca donde difieren


In [ ]:
# TODO 5.2.2: Muestra los términos más importantes por clase
# Hint: modelo.coef_[0] contiene los coeficientes; mapéalos a feature names del TF-IDF


**Tu análisis de errores:**

*[¿Por qué crees que el modelo se equivoca en esos casos? ¿Qué patrones observas?]*

---

## Parte 6: Preguntas de Reflexión (10 pts)

Responde las siguientes preguntas con al menos 3-4 oraciones cada una.

### Pregunta 1 (3 pts)

¿Por qué es importante usar `class_weight='balanced'` cuando el dataset está desbalanceado? ¿Qué pasaría si no lo usáramos? Explica con referencia a tus resultados.

**Tu respuesta:**

*[Escribe aquí]*

### Pregunta 2 (3 pts)

¿Por qué es incorrecto hacer `fit` del TF-IDF con los datos de test? ¿Qué problema introduciría esto? Relaciona tu respuesta con el concepto de *data leakage*.

**Tu respuesta:**

*[Escribe aquí]*

### Pregunta 3 (4 pts)

Compara la representación TF-IDF con Bag of Words (BoW) simple. ¿Qué ventajas tiene TF-IDF sobre BoW para esta tarea? Menciona al menos una limitación que comparten **ambos** métodos que podría ser superada por representaciones más avanzadas como Word2Vec (que veremos en la Semana 4).

**Tu respuesta:**

*[Escribe aquí]*

---

## Bonus (hasta +10 pts extra)

Implementa **una** de las siguientes ideas:

1. **Comparación BoW vs TF-IDF** (+5 pts): Entrena el mismo modelo usando `CountVectorizer` (BoW) y compara con TF-IDF. ¿Cuál da mejores resultados?

2. **Predicción interactiva** (+5 pts): Crea una función `predecir_sentimiento(texto)` que reciba una reseña nueva en español y devuelva la predicción con probabilidad.

3. **Análisis de n-gramas** (+5 pts): Entrena modelos con unigrams, bigrams y trigrams por separado. ¿Cuál funciona mejor? ¿Por qué?

In [ ]:
# BONUS (opcional)


---

## Instrucciones de Entrega

1. **Formato:** Entrega este notebook (`.ipynb`) con todas las celdas ejecutadas
2. **Código:** Todo el código debe ser ejecutable de principio a fin sin errores
3. **Respuestas:** Todas las preguntas de texto deben estar respondidas
4. **Nombre del archivo:** `tarea01_APELLIDO_NOMBRE.ipynb`

---

*Procesamiento de Lenguaje Natural — UCB 2026*